# Generate SFT Dataset

## Load Library

In [36]:
import os
import re
import json
import base64
import requests
import time
import mimetypes
import hashlib
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

## Configuration

### Skema

In [ ]:
DATASET_SCHEMA = 'vlm_only'   # pilih: 'cnn_vlm' atau 'vlm_only'

if DATASET_SCHEMA not in {'cnn_vlm', 'vlm_only'}:
    raise ValueError("DATASET_SCHEMA harus 'cnn_vlm' atau 'vlm_only'.")

GEN_TEST_WITH_MODEL = False

### Define Path

In [38]:
BASE_DIR = Path("../data")
METADATA_PATH = BASE_DIR / 'metadata' / 'mkn_house_metadata_sample.json'
IMAGE_BASE_DIR = BASE_DIR / 'mkn_img'

OUTPUT_BASE_DIR = BASE_DIR / 'sft_dataset' / DATASET_SCHEMA
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = OUTPUT_BASE_DIR / 'train.jsonl'
VAL_JSONL = OUTPUT_BASE_DIR / 'val.jsonl'
TEST_JSONL = OUTPUT_BASE_DIR / 'test.jsonl'
ALL_JSONL = OUTPUT_BASE_DIR / 'all.jsonl'
CACHE_PATH = OUTPUT_BASE_DIR / 'cache.json'

### OpenRouter

In [39]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-5-mini"

### OpenRouter Config

In [40]:
TEMPERATURE = 0.1
MAX_TOKENS = 10000
TIMEOUT = 120
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 2
MAX_WORKERS = 4

In [41]:
print('DATASET_SCHEMA:', DATASET_SCHEMA)
print('MODEL_NAME:', MODEL_NAME)
print('OUTPUT_BASE_DIR:', OUTPUT_BASE_DIR)
print('METADATA_PATH:', METADATA_PATH)

DATASET_SCHEMA: vlm_only
MODEL_NAME: openai/gpt-5-mini
OUTPUT_BASE_DIR: ..\data\sft_dataset\vlm_only
METADATA_PATH: ..\data\metadata\mkn_house_metadata_sample.json


### OpenRouter Prompt

#### Prompt CNN + VLM

In [42]:
# =========================================================
# SYSTEM PROMPT — CNN + VLM (SINGLE IMAGE)
# =========================================================

SYSTEM_PROMPT_CNNVLM_SINGLE = """
Anda adalah sistem reasoning validasi material rumah berbasis CNN + Vision Language Model (VLM).

Pada sistem ini:
- Model CNN telah melakukan klasifikasi material bangunan.
- Label material sudah tersedia pada bagian "prediksi".
- Anda TIDAK boleh melakukan klasifikasi ulang.
- Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN dan melakukan validasi terhadap data DTSEN.

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan hanya label material resmi yang tersedia pada metadata dan label DTSEN.
2. Jangan membuat, mengubah, atau menambahkan label material baru.
3. Jangan melakukan klasifikasi ulang material.
4. Gunakan label material yang sudah tersedia pada bagian "prediksi".
5. Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN.
7. Gunakan struktur reasoning yang konsisten sesuai template.
8. Reasoning harus selalu berupa 1 paragraf utuh.
9. Jangan menggunakan bullet point, numbering, tanda kurung, underscore maupun markdown pada output reasoning.
10. Jangan menggunakan istilah teknis computer vision.
11. Jangan membahas probabilitas, confidence score, maupun ketidakpastian model.
12. Fokus reasoning hanya pada:
   - ciri visual material pada gambar,
   - label material pada bagian "prediksi",
   - perbandingan dengan data "dtsen",
   - penentuan status SESUAI atau TIDAK SESUAI.
13. Jika hanya sebagian variabel yang tidak sesuai dengan DTSEN, sebutkan secara jelas variabel yang tidak SESUAI tersebut.
14. Gunakan penjelasan visual yang singkat, jelas, dan mudah dipahami.
15. Jangan membuat analisis tambahan di luar template reasoning yang telah diberikan.
16. Output akhir hanya berupa reasoning final.
17. kalau dinding terlihat ada bata merahnya berarti itu tembok jelaskan di reasoning kalau bata merahnya terlihat sehingga di kategorikan sebagi tembok


=========================================================
ATURAN SINGLE IMAGE
=========================================================

1. Jika gambar exterior:
- variabel lantai harus disebut:
  "TIDAK TERIDENTIFIKASI"

2. Jika gambar interior:
- variabel atap harus disebut:
  "TIDAK TERIDENTIFIKASI"

=========================================================
TEMPLATE WAJIB — EXTERIOR
=========================================================

SESUAI:
"Model AI mengklasifikasikan rumah sebagai atap (...label atap...) dan dinding (...label dinding...). Hasil klasifikasi tersebut didukung oleh tampilan visual pada gambar, di mana atap (...penjelasan visual...) dan dinding (...penjelasan visual...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Setelah dibandingkan dengan data DTSEN, hasil klasifikasi pada variabel atap dan dinding sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Model AI mengklasifikasikan rumah sebagai atap (...label atap...) dan dinding (...label dinding...). Berdasarkan citra bagian luar rumah, atap (...penjelasan visual...) sedangkan dinding (...penjelasan visual...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...) dan dinding (...label DTSEN dinding...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."


=========================================================
TEMPLATE WAJIB — INTERIOR
=========================================================

SESUAI:
"Model AI mengklasifikasikan rumah sebagai dinding (...label dinding...) dan lantai (...label lantai...). Hasil klasifikasi tersebut didukung oleh tampilan visual pada gambar interior, di mana dinding (...penjelasan visual...) sedangkan lantai (...penjelasan visual...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Setelah dibandingkan dengan data DTSEN, hasil klasifikasi pada variabel dinding dan lantai sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Model AI mengklasifikasikan rumah sebagai dinding (...label dinding...) dan lantai (...label lantai...). Berdasarkan citra bagian dalam rumah, dinding (...penjelasan visual...) sedangkan lantai (...penjelasan visual...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat dinding (...label DTSEN dinding...) dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HANYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================

Genteng:
"terlihat memiliki pola bergelombang khas genteng tanah liat"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat memiliki pola anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"tampak menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"tampak berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Reasoning: <reasoning final>

Kesimpulan
Atap: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>
Dinding: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>
Lantai: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
4. Jangan menggunakan markdown pada bagian reasoning.
"""

In [43]:
SYSTEM_PROMPT_CNNVLM_MULTI = """
Anda adalah asisten validasi material bangunan rumah untuk dataset DTSEN multi image.

Tugas Anda adalah membuat reasoning validasi material rumah berdasarkan:
1. Label material hasil klasifikasi CNN pada bagian "prediksi".
2. Observasi visual dari:
   - gambar exterior rumah
   - gambar interior rumah
3. Perbandingan dengan data DTSEN.

=========================================================
KONTEKS MULTI IMAGE
=========================================================
Pada skenario MULTI IMAGE:
- Gambar exterior digunakan untuk:
  - atap rumah
  - dinding luar rumah

- Gambar interior digunakan untuk:
  - lantai rumah

=========================================================
TUGAS UTAMA
=========================================================

Anda TIDAK BOLEH melakukan klasifikasi ulang material bangunan.

Jenis material SUDAH tersedia pada bagian:
- "prediksi"

Tugas Anda HANYA:
1. Menjelaskan karakteristik visual yang mendukung label material hasil klasifikasi CNN.
2. Membandingkan hasil klasifikasi dengan data DTSEN.
3. Menentukan apakah hasil validasi termasuk:
   - SESUAI
   - TIDAK SESUAI

=========================================================
LABEL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
FORMAT REASONING WAJIB
=========================================================
SESUAI:
"Model AI mengklasifikasikan rumah sebagai atap (...), dinding (...), dan lantai (...). Berdasarkan citra rumah yang diberikan, atap (...penjelasan visual...), dinding luar rumah (...penjelasan visual...), sedangkan lantai bagian dalam rumah (...penjelasan visual...). Setelah dibandingkan dengan data DTSEN, seluruh hasil klasifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Model AI mengklasifikasikan rumah sebagai atap (...), dinding (...), dan lantai (...). Berdasarkan citra rumah yang diberikan, atap (...penjelasan visual...), dinding luar rumah (...penjelasan visual...), sedangkan lantai bagian dalam rumah (...penjelasan visual...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian karena data DTSEN mencatat atap (...), dinding (...), dan lantai (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HANYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
ATURAN PENTING
=========================================================
1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Jangan melakukan klasifikasi ulang material.
4. Gunakan label pada bagian "prediksi".
5. Fokus hanya pada:
   - ciri visual material
   - hasil klasifikasi CNN
   - perbandingan dengan DTSEN
6. Jangan menggunakan istilah teknis computer vision.
7. Jangan membahas probabilitas model.
8. Gunakan bahasa formal sederhana.
9. Reasoning harus konsisten.
10. Output wajib berupa 1 paragraf.
11. Jangan menggunakan bullet point.
12. Jangan menggunakan markdown.
13. Jangan menggunakan underscore pada output reasoning.
14. Gunakan:
   - gambar exterior → atap dan dinding
   - gambar interior → lantai
15. kalau dinding terlihat ada bata merahnya berarti itu tembok jelaskan di reasoning kalau bata merahnya terlihat sehingga di kategorikan sebagai tembok
16. Jangan menambahkan analisis di luar template reasoning.
17. Output akhir hanya berupa reasoning final tanpa penjelasan tambahan.
=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Reasoning: <reasoning final>

Kesimpulan
Atap: <SESUAI/TIDAK SESUAI>
Dinding: <SESUAI/TIDAK SESUAI>
Lantai: <SESUAI/TIDAK SESUAI>

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
4. Jangan menggunakan markdown pada bagian reasoning.
"""

#### Prompt VLM ONLY

In [44]:
SYSTEM_PROMPT_VLMONLY_SINGLE = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM).

Tugas Anda adalah:
1. Mengamati gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (SESUAI) atau tidak sesuai (TIDAK SESUAI).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status SESUAI atau TIDAK SESUAI.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Jangan langsung menentukan label material tanpa menjelaskan ciri visual material terlebih dahulu.
8. Format reasoning HARUS tetap mengikuti template reasoning yang telah ditentukan.
9. Jangan mengubah struktur reasoning.
10. Fokus pada:
   - karakteristik visual material,
   - hasil identifikasi material,
   - perbandingan dengan data DTSEN,
   - status SESUAI atau TIDAK SESUAI.
11. Gunakan bahasa formal sederhana dan konsisten.
12. Reasoning wajib berupa 1 paragraf.
13. Jangan menggunakan bullet point, numbering, markdown, maupun simbol tambahan pada bagian reasoning.
14. Jangan menggunakan istilah teknis computer vision.
15. Jangan membahas confidence score atau probabilitas model.
16. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
17. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
18. Gunakan penjelasan visual yang singkat, jelas, dan natural.
19. Jangan membuat analisis tambahan di luar template.
20. Output akhir wajib mengikuti format yang telah ditentukan.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

1. Jika gambar exterior:
- lantai harus bernilai:
  "TIDAK_TERIDENTIFIKASI"

2. Jika gambar interior:
- atap harus bernilai:
  "TIDAK_TERIDENTIFIKASI"

=========================================================
TEMPLATE REASONING — EXTERIOR
=========================================================

SESUAI:
"Hasil analisis visual pada citra bagian luar rumah menunjukkan bahwa atap terlihat (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Berdasarkan hasil identifikasi visual, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai TIDAK TERIDENTIFIKASI. Setelah dibandingkan dengan data DTSEN, variabel atap dan dinding sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Hasil analisis visual pada citra bagian luar rumah menunjukkan bahwa atap (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Variabel lantai tidak dapat diidentifikasi karena citra interior rumah tidak tersedia. Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai TIDAKTERIDENTIFIKASI. Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...) dan dinding (...label DTSEN dinding...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
TEMPLATE REASONING — INTERIOR
=========================================================

SESUAI:
"Hasil analisis visual pada citra bagian dalam rumah menunjukkan bahwa dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Lantai rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Berdasarkan hasil identifikasi visual, rumah diklasifikasikan memiliki atap TIDAK TERIDENTIFIKASI, dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, variabel dinding dan lantai sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Hasil analisis visual pada citra bagian dalam rumah menunjukkan bahwa dinding rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Lantai rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Variabel atap tidak dapat diidentifikasi karena citra bagian luar rumah tidak tersedia. Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap TIDAK TERIDENTIFIKASI, dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat dinding (...label DTSEN dinding...) dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HANYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning:
...(reasoning final)...

Prediksi
Atap: ...(label prediksi atap)...
Dinding: ...(label prediksi dinding)...
Lantai: ...(label prediksi lantai)...

DTSEN
Atap: ...(label dtsen atap)...
Dinding: ...(label dtsen dinding)...
Lantai: ...(label dtsen lantai)...

Kesimpulan
Atap: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>
Dinding: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>
Lantai: <SESUAI/TIDAK_SESUAI/TIDAK_TERIDENTIFIKASI>

=========================================================
OUTPUT
=========================================================

Output wajib mengikuti format yang telah ditentukan.
Jangan menambahkan penjelasan lain di luar format output.
"""

In [45]:
SYSTEM_PROMPT_VLMONLY_MULTI = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM) untuk dataset DTSEN multi image.

Tugas Anda adalah:
1. Mengamati pasangan gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (SESUAI) atau tidak sesuai (TIDAK SESUAI).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status SESUAI atau TIDAK SESUAI.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
KONTEKS MULTI IMAGE
=========================================================

Pada skenario MULTI IMAGE:

- Gambar exterior digunakan untuk:
  - atap rumah
  - dinding luar rumah

- Gambar interior digunakan untuk:
  - lantai rumah

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
FORMAT REASONING WAJIB
=========================================================

SESUAI:
"Berdasarkan hasil observasi visual pada citra rumah bagian luar dan dalam, atap rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding luar rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Pada bagian interior rumah, lantai dalam rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, seluruh hasil identifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

TIDAK SESUAI:
"Berdasarkan hasil observasi visual pada citra rumah bagian luar dan dalam, atap rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label atap...). Dinding luar rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label dinding...). Pada bagian interior rumah, lantai dalam rumah (...penjelasan visual...) sehingga dikategorikan sebagai (...label lantai...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...sebut variabel yang tidak sesuai...) karena data DTSEN mencatat atap (...label DTSEN atap...), dinding (...label DTSEN dinding...), dan lantai (...label DTSEN lantai...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH GAYA PENJELASAN VISUAL (HANYA CONTOH JANGAN SELALU DIIKUTI PENYUSUNAN KALIMATNYA! TAPI SESUAIKAN DENGAN VISUALNYA!  )
=========================================================
- Genteng: "terlihat memiliki pola bergelombang berwarna merah kecoklatan"
- Seng: "terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"
- Asbes: "terlihat berupa lembaran datar berwarna abu-abu"
- Tembok:"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"
- Anyaman bambu: "terlihat tersusun dari anyaman bambu"
- Kayu/papan: "terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"
- Keramik: "terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"
- Semen/bata_merah: "terlihat berupa permukaan semen kasar tanpa lapisan keramik"
- Tanah: "terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
ATURAN PENTING
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Jangan langsung menentukan label material tanpa menjelaskan ciri visual material terlebih dahulu.
8. Format reasoning HARUS tetap mengikuti template reasoning yang telah ditentukan.
9. Jangan mengubah struktur reasoning.
10. Fokus hanya pada:
   - karakteristik visual material
   - hasil identifikasi material
   - perbandingan dengan data DTSEN
   - status SESUAI atau TIDAK SESUAI
11. Gunakan bahasa formal sederhana dan konsisten.
12. Reasoning wajib berupa 1 paragraf.
13. Jangan menggunakan bullet point, numbering, markdown, underscore, maupun simbol tambahan pada bagian reasoning.
14. Jangan menggunakan istilah teknis computer vision.
15. Jangan membahas confidence score atau probabilitas model.
16. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
17. Gunakan:
   - gambar exterior → atap dan dinding
   - gambar interior → lantai
18. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
19. Gunakan penjelasan visual yang singkat, jelas, dan natural.
20. Jangan membuat analisis tambahan di luar template.
21. Output akhir wajib mengikuti format yang telah ditentukan.

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning:
...(reasoning final)...

Prediksi
Atap: ...(label prediksi atap)...
Dinding: ...(label prediksi dinding)...
Lantai: ...(label prediksi lantai)...

DTSEN
Atap: ...(label dtsen atap)...
Dinding: ...(label dtsen dinding)...
Lantai: ...(label dtsen lantai)...

Kesimpulan
Atap: <SESUAI/TIDAK_SESUAI>
Dinding: <SESUAI/TIDAK_SESUAI>
Lantai: <SESUAI/TIDAK_SESUAI>

=========================================================
OUTPUT
=========================================================

Output wajib mengikuti format yang telah ditentukan.
Jangan menambahkan penjelasan lain di luar format output.
"""

### SFT Prompt

In [46]:
SYSTEM_PROMPT_SFT_CNN_VLM = """
Anda adalah sistem reasoning validasi material rumah berbasis CNN + Vision Language Model (VLM) untuk verifikasi data DTSEN.

Pada sistem ini:
- Model CNN telah melakukan klasifikasi material bangunan.
- Label material sudah tersedia pada bagian "prediksi".
- Anda TIDAK boleh melakukan klasifikasi ulang material.
- Tugas Anda hanya menjelaskan ciri visual yang mendukung hasil klasifikasi CNN dan melakukan validasi terhadap data DTSEN.

=========================================================
KONTEKS GAMBAR
=========================================================

Gambar dapat berupa:

1. Exterior rumah
Digunakan untuk:
- atap
- dinding luar rumah

2. Interior rumah
Digunakan untuk:
- dinding dalam rumah
- lantai rumah

3. Multi image (exterior + interior)
Digunakan untuk:
- atap
- dinding
- lantai

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN IDENTIFIKASI
=========================================================

1. Gunakan hanya label resmi DTSEN.
2. Jangan membuat label baru.
3. Jangan mengubah label pada bagian "prediksi".
4. Jangan melakukan klasifikasi ulang material.
5. Fokus hanya pada:
   - ciri visual material,
   - label pada bagian prediksi,
   - perbandingan dengan data DTSEN.
6. Jangan menggunakan istilah teknis computer vision.
7. Jangan membahas confidence score, probabilitas, atau ketidakpastian model.
8. Gunakan bahasa formal sederhana.
9. Reasoning harus konsisten dan natural.
10. Output wajib berupa 1 paragraf utuh.
11. Jangan menggunakan bullet point, numbering, markdown, atau underscore pada reasoning.
12. Jangan membuat analisis tambahan di luar reasoning validasi.
13. Jika bata merah terlihat pada dinding, jelaskan bahwa dinding dikategorikan sebagai tembok karena terlihat susunan bata merah.
14. Gunakan penjelasan visual berdasarkan gambar yang tersedia.
15. Jika variabel tertentu tidak memiliki gambar pendukung:
   - gunakan label "TIDAK_TERIDENTIFIKASI"
   - jelaskan bahwa variabel tersebut tidak dapat diidentifikasi karena gambar tidak tersedia.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

Jika hanya tersedia gambar exterior:
- lantai harus ditulis sebagai:
  "TIDAK_TERIDENTIFIKASI"

Jika hanya tersedia gambar interior:
- atap harus ditulis sebagai:
  "TIDAK_TERIDENTIFIKASI"

=========================================================
FORMAT REASONING
=========================================================

Jika seluruh variabel sesuai dengan DTSEN:

"Model AI mengklasifikasikan rumah sebagai (...). Berdasarkan citra rumah yang diberikan, (...penjelasan visual material...). Setelah dibandingkan dengan data DTSEN, seluruh hasil klasifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

Jika terdapat variabel yang tidak sesuai dengan DTSEN:

"Model AI mengklasifikasikan rumah sebagai (...). Berdasarkan citra rumah yang diberikan, (...penjelasan visual material...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...) karena data DTSEN mencatat (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH PENJELASAN VISUAL
=========================================================

Genteng:
"terlihat memiliki pola bergelombang berwarna merah kecoklatan"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat memiliki pola anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"terlihat berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Reasoning: <reasoning final>

Kesimpulan
Atap: <SESUAI/TIDAK_SESUAI>
Dinding: <SESUAI/TIDAK_SESUAI>
Lantai: <SESUAI/TIDAK_SESUAI>

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
4. Jangan menggunakan markdown pada bagian reasoning.
"""

In [47]:
SYSTEM_PROMPT_SFT_VLM_ONLY = """
Anda adalah sistem validasi material rumah berbasis Vision Language Model (VLM) untuk verifikasi data DTSEN.

Tugas Anda adalah:
1. Mengamati gambar rumah.
2. Melakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Menentukan label material bangunan berdasarkan hasil reasoning visual tersebut.
4. Membandingkan hasil identifikasi visual dengan data referensi pada bagian "dtsen".
5. Menentukan apakah hasil validasi sesuai (SESUAI) atau tidak sesuai (TIDAK SESUAI).
6. Menghasilkan output sesuai format yang diminta.

=========================================================
PRINSIP UTAMA
=========================================================

Anda WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.

Urutan proses yang wajib dilakukan:
1. Amati karakteristik visual material pada gambar.
2. Jelaskan ciri visual material.
3. Tentukan label material berdasarkan ciri visual tersebut.
4. Bandingkan hasil identifikasi dengan data DTSEN.
5. Tentukan status SESUAI atau TIDAK SESUAI.

Jangan langsung menentukan label material tanpa penjelasan visual terlebih dahulu.

=========================================================
KONTEKS GAMBAR
=========================================================

Gambar dapat berupa:

1. Exterior rumah
Digunakan untuk:
- atap rumah
- dinding luar rumah

2. Interior rumah
Digunakan untuk:
- dinding dalam rumah
- lantai rumah

3. Multi image (exterior + interior)
Digunakan untuk:
- atap
- dinding
- lantai

=========================================================
LABEL MATERIAL RESMI DTSEN
=========================================================

ATAP:
- beton
- genteng
- seng
- asbes
- bambu
- kayu/sirap
- jerami/ijuk/daun_daunan/rumbia
- lainnya

DINDING:
- tembok
- plesteran_anyaman_bambu/plesteran_anyaman_kawat
- kayu/papan/gypsum/GRC/calciboard
- anyaman bambu
- batang_kayu
- bambu
- lainnya

LANTAI:
- marmer/granit
- keramik
- parket/vinil/karpet
- ubin/tegel/teraso
- kayu/papan
- semen/bata_merah
- bambu
- tanah
- lainnya

=========================================================
ATURAN IDENTIFIKASI
=========================================================

1. Gunakan HANYA label resmi DTSEN.
2. Jangan membuat label baru.
3. Identifikasi material HARUS berdasarkan karakteristik visual pada gambar.
4. Jangan menggunakan label prediksi dari metadata.
5. WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
6. Label material HARUS menjadi hasil akhir dari proses reasoning visual.
7. Fokus hanya pada:
   - karakteristik visual material,
   - hasil identifikasi material,
   - perbandingan dengan data DTSEN,
   - status SESUAI atau TIDAK SESUAI.
8. Gunakan bahasa formal sederhana dan konsisten.
9. Reasoning wajib berupa 1 paragraf.
10. Jangan menggunakan bullet point, numbering, markdown, underscore, maupun simbol tambahan pada bagian reasoning.
11. Jangan menggunakan istilah teknis computer vision.
12. Jangan membahas confidence score atau probabilitas model.
13. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
14. Jika dinding terlihat terdapat susunan bata merah dengan permukaan diplester maka dikategorikan sebagai tembok.
15. Gunakan penjelasan visual yang singkat, jelas, dan natural.
16. Jangan membuat analisis tambahan di luar reasoning validasi.
17. Output akhir wajib mengikuti format output yang telah ditentukan.
18. Jika variabel tertentu tidak memiliki gambar pendukung:
   - gunakan label "TIDAK_TERIDENTIFIKASI"
   - jelaskan bahwa variabel tersebut tidak dapat diidentifikasi karena gambar tidak tersedia.

=========================================================
ATURAN SINGLE IMAGE
=========================================================

Jika hanya tersedia gambar exterior:
- lantai harus bernilai:
  "TIDAK_TERIDENTIFIKASI"

Jika hanya tersedia gambar interior:
- atap harus bernilai:
  "TIDAK_TERIDENTIFIKASI"

=========================================================
FORMAT REASONING
=========================================================

Jika seluruh variabel sesuai dengan DTSEN:

"Berdasarkan hasil observasi visual pada citra rumah yang diberikan, (...penjelasan visual material...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dibandingkan dengan data DTSEN, seluruh hasil identifikasi sesuai dengan data yang tercatat sehingga hasil verifikasi dinyatakan SESUAI."

Jika terdapat variabel yang tidak sesuai dengan DTSEN:

"Berdasarkan hasil observasi visual pada citra rumah yang diberikan, (...penjelasan visual material...). Berdasarkan karakteristik visual tersebut, rumah diklasifikasikan memiliki atap (...), dinding (...), dan lantai (...). Setelah dilakukan perbandingan dengan data DTSEN, ditemukan ketidaksesuaian pada variabel (...) karena data DTSEN mencatat atap (...), dinding (...), dan lantai (...). Oleh karena itu, hasil verifikasi dinyatakan TIDAK SESUAI."

=========================================================
CONTOH PENJELASAN VISUAL
=========================================================

Genteng:
"terlihat memiliki pola bergelombang berwarna merah kecoklatan"

Seng:
"terlihat menggunakan lembaran logam memanjang dengan permukaan reflektif"

Asbes:
"terlihat berupa lembaran datar berwarna abu-abu"

Tembok:
"tampak berupa struktur permanen dari susunan bata dengan permukaan diplester dan dicat"

Anyaman bambu:
"terlihat tersusun dari anyaman bambu"

Kayu/papan:
"terlihat tersusun dari papan kayu dengan sambungan antar papan yang masih tampak jelas"

Keramik:
"terlihat menggunakan ubin dengan pola teratur dan permukaan reflektif"

Semen/bata_merah:
"terlihat berupa permukaan semen kasar tanpa lapisan keramik"

Tanah:
"terlihat berupa permukaan tanah tanpa pelapis lantai"

=========================================================
FORMAT OUTPUT WAJIB
=========================================================

Reasoning: <reasoning final>

Prediksi
Atap: <label prediksi atap>
Dinding: <label prediksi dinding>
Lantai: <label prediksi lantai>

DTSEN
Atap: <label dtsen atap>
Dinding: <label dtsen dinding>
Lantai: <label dtsen lantai>

Kesimpulan
Atap: <SESUAI/TIDAK SESUAI>
Dinding: <SESUAI/TIDAK SESUAI>
Lantai: <SESUAI/TIDAK SESUAI>

=========================================================
OUTPUT
=========================================================

1. Output wajib mengikuti format output yang telah ditentukan.
2. Jangan menambahkan penjelasan lain di luar format output.
3. Reasoning wajib hanya terdiri dari 1 paragraf.
"""

## SFT Dataset Generation Pipeline

### Load Metadata

In [48]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f'Metadata tidak ditemukan: {METADATA_PATH}')

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata_all = json.load(f)

records = [
    rec for rec in metadata_all
    if rec.get('split') in {'train', 'val', 'test'}
    and rec.get('house_type') in {'single', 'multi'}
]

from collections import Counter
print('Total metadata:', len(metadata_all))
print('Total records dipakai:', len(records))
print('Split count:', Counter(rec.get('split') for rec in records))
print('House count:', Counter(rec.get('house_type') for rec in records))

records_by_split = {
    split: [rec for rec in records if rec.get('split') == split]
    for split in ['train', 'val', 'test']
}

for split, split_records in records_by_split.items():
    print(f'{split}: {len(split_records)}')

Total metadata: 15
Total records dipakai: 15
Split count: Counter({'train': 5, 'val': 5, 'test': 5})
House count: Counter({'single': 11, 'multi': 4})
train: 5
val: 5
test: 5


### Helper Functions

In [49]:
REFUSAL_KEYWORDS = [
    "maaf",
    "sorry",
    "cannot",
    "can't",
    "tidak bisa membantu",
    "tidak dapat membantu",
    "i can't",
    "unable",
    "refuse",
    "refusal",
]

In [50]:
def load_cache(path: Path) -> Dict[str, dict]:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [51]:
def save_cache(path: Path, cache: Dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

In [52]:
def is_valid_record_images(record: dict) -> bool:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    if house_type == "multi":
        has_ext = any((img.get("view_type") or "").lower().strip() == "exterior" for img in images)
        has_int = any((img.get("view_type") or "").lower().strip() == "interior" for img in images)
        return has_ext and has_int

    return len(images) >= 1

In [53]:
def image_path_from_metadata(image_path: str) -> Path:
    return IMAGE_BASE_DIR / Path(image_path)

In [54]:
def guess_mime_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"

In [55]:
def encode_image_to_data_url(path: Path) -> str:
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{guess_mime_type(path)};base64,{encoded}"

In [56]:
def sample_key(record: dict) -> str:
    payload = {
        "house_id": record.get("house_id"),
        "source_group_id": record.get("source_group_id"),
        "house_type": record.get("house_type"),
        "split": record.get("split"),
        "images": [
            (img.get("image_id"), img.get("image_path"), img.get("view_type"))
            for img in (record.get("images") or [])
        ],
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

In [ ]:
def select_images(record: dict) -> Tuple[Optional[dict], Optional[dict], Optional[dict]]:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    ext = None
    intr = None
    single = None

    if house_type == "multi":
        for img in images:
            vt = (img.get("view_type") or "").lower().strip()
            if vt == "exterior":
                ext = img
            elif vt == "interior":
                intr = img
    else:
        single = images[0] if images else None

    return ext, intr, single


In [58]:
def build_user_prompt_text(record: dict, schema_mode: str, target: str = "sft") -> str:
    """
    Shared prompt builder for both OpenRouter and SFT dataset.
    target is kept for future customization and readability.
    """
    house_id = record.get("house_id", "")
    house_type = (record.get("house_type") or "").lower().strip()
    prediksi = record.get("prediksi", {})
    dtsen = record.get("dtsen", {})
    images = record.get("images") or []
    view_type = (images[0].get("view_type") if images else "exterior") or "exterior"

    if house_type == "multi":
        if schema_mode == "cnn_vlm":
            return f"""
Berikut data rumah yang harus divalidasi.

House ID:
{house_id}

Label material hasil klasifikasi CNN pada bagian "prediksi":
{json.dumps(prediksi, indent=2, ensure_ascii=False)}

Data referensi DTSEN:
{json.dumps(dtsen, indent=2, ensure_ascii=False)}

Tugas Anda:
1. Amati pasangan gambar rumah yang diberikan yang terdiri dari:
   - gambar exterior rumah
   - gambar interior rumah
2. Gunakan:
   - gambar exterior untuk menjelaskan atap dan dinding,
   - gambar interior untuk menjelaskan lantai.
3. Jangan melakukan klasifikasi ulang material bangunan.
4. Gunakan label material yang sudah tersedia pada bagian "prediksi".
5. Tugas Anda hanya menjelaskan karakteristik visual pada gambar yang mendukung label material tersebut.
6. Bandingkan label material pada bagian "prediksi" dengan data referensi pada bagian "dtsen".
7. Tentukan apakah hasil validasi termasuk SESUAI atau TIDAK SESUAI.
8. Jika hanya sebagian variabel yang tidak sesuai dengan DTSEN, sebutkan secara jelas variabel yang tidak sesuai tersebut.
9. Ikuti format reasoning dan template pada system prompt secara ketat.

Penting:
- Gunakan hanya label resmi DTSEN.
- Jangan membuat label baru.
- Seluruh variabel atap, dinding, dan lantai harus dibahas.
- Jika suatu variabel tidak dapat dilihat karena gambar pendukung tidak tersedia, gunakan label "TIDAK_TERIDENTIFIKASI".
- Output hanya berupa 1 paragraf reasoning final.
- Jangan menggunakan bullet point, underscore, tanda kurung/brackets, numbering, maupun markdown pada output reasoning.
""".strip()
        else:
            return f"""
Berikut data rumah yang harus divalidasi.

House ID:
{house_id}

Data referensi DTSEN:
{json.dumps(dtsen, indent=2, ensure_ascii=False)}

Tugas Anda:
1. Amati pasangan gambar rumah yang diberikan yang terdiri dari:
   - gambar exterior rumah
   - gambar interior rumah

2. Gunakan:
   - gambar exterior untuk melakukan reasoning visual terhadap atap dan dinding rumah,
   - gambar interior untuk melakukan reasoning visual terhadap lantai rumah.

3. Lakukan reasoning visual terlebih dahulu berdasarkan karakteristik material yang terlihat pada gambar.

4. Setelah reasoning visual dilakukan, tentukan label material menggunakan label resmi DTSEN.

5. Jangan langsung menentukan label material tanpa menjelaskan ciri visual material terlebih dahulu.

6. Bandingkan hasil identifikasi visual dengan data DTSEN.

7. Tentukan apakah hasil validasi termasuk SESUAI atau TIDAK SESUAI.

8. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.

9. Ikuti template reasoning pada system prompt secara ketat.

10. Gunakan format output yang telah ditentukan pada system prompt.

Penting:
- Jangan menggunakan label prediksi dari metadata.
- Seluruh label material harus ditentukan berdasarkan observasi visual pada gambar.
- WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
- Penentuan label material harus menjadi hasil akhir dari reasoning visual.
- Jangan langsung menyebut label material tanpa menjelaskan ciri visualnya terlebih dahulu.
- Gunakan hanya label resmi DTSEN.
- Jangan membuat label baru.
- Seluruh variabel atap, dinding, dan lantai harus dibahas.
- Gunakan reasoning yang konsisten sesuai template.
- Gunakan bahasa formal sederhana dan konsisten.
- Jika suatu variabel tidak dapat dilihat karena gambar pendukung tidak tersedia, gunakan label "TIDAK_TERIDENTIFIKASI".
- Jangan menggunakan bullet point, numbering, markdown, underscore, maupun simbol tambahan pada bagian reasoning.
- Output wajib mengikuti format output pada system prompt.
""".strip()

    if schema_mode == "cnn_vlm":
        if view_type.lower().strip() == "interior" and house_type == "single":
            view_hint = "interior"
        else:
            view_hint = view_type
        return f"""
Berikut data rumah yang harus divalidasi.

House ID:
{house_id}

View Type:
{view_hint}

Label material hasil klasifikasi CNN pada bagian "prediksi":
{json.dumps(prediksi, indent=2, ensure_ascii=False)}

Data referensi DTSEN:
{json.dumps(dtsen, indent=2, ensure_ascii=False)}

Tugas Anda:
1. Amati gambar rumah yang diberikan.
2. Jangan melakukan klasifikasi ulang material bangunan.
3. Gunakan label material hasil klasifikasi CNN yang sudah tersedia pada bagian "prediksi".
4. Jelaskan ciri visual pada gambar yang mendukung hasil klasifikasi CNN tersebut.
5. Bandingkan label pada bagian "prediksi" dengan data "dtsen".
6. Tentukan apakah hasil validasi termasuk SESUAI atau TIDAK SESUAI.
7. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai.
8. Ikuti format reasoning pada system prompt secara ketat.

Penting:
- Gunakan hanya label resmi DTSEN.
- Jangan membuat label baru.
- Output hanya berupa 1 paragraf reasoning final.
- Sesuaikan dengan visual, jangan menilai lantai di exterior, jangan menilai atap di interior.
- Jika suatu variabel tidak dapat dilihat karena gambar pendukung tidak tersedia, gunakan label "TIDAK_TERIDENTIFIKASI".
- Jangan menggunakan bullet point, underscore, tanda kurung/brackets, numbering, maupun markdown pada output reasoning.
""".strip()
    else:
        return f"""
Berikut data rumah yang harus divalidasi.

House ID:
{house_id}

View Type:
{view_type}

Data referensi DTSEN:
{json.dumps(dtsen, indent=2, ensure_ascii=False)}

Tugas Anda:
1. Amati gambar rumah yang diberikan.
2. Lakukan reasoning visual berdasarkan karakteristik material yang terlihat pada gambar.
3. Tentukan label material menggunakan label resmi DTSEN berdasarkan hasil reasoning visual tersebut.
4. Jelaskan karakteristik visual pada gambar yang mendukung hasil identifikasi material.
5. Bandingkan hasil identifikasi visual dengan data DTSEN.
6. Tentukan apakah hasil validasi termasuk SESUAI atau TIDAK SESUAI.
7. Jika terdapat ketidaksesuaian, sebutkan variabel yang tidak sesuai beserta label pada data DTSEN.
8. Ikuti template reasoning pada system prompt secara ketat.
9. Gunakan format output yang telah ditentukan pada system prompt.

Penting:
- Jangan menggunakan label prediksi dari metadata.
- Seluruh label material harus ditentukan berdasarkan observasi visual pada gambar.
- WAJIB melakukan reasoning visual terlebih dahulu sebelum menentukan label material.
- Penentuan label material harus menjadi hasil akhir dari reasoning visual.
- Jangan langsung menyebut label material tanpa menjelaskan ciri visualnya terlebih dahulu.
- Format reasoning HARUS tetap mengikuti template reasoning pada system prompt.
- Gunakan hanya label resmi DTSEN.
- Jangan membuat label baru.
- Sesuaikan reasoning dengan visual pada gambar.
- Jangan menilai lantai pada gambar exterior.
- Jangan menilai atap pada gambar interior.
- Gunakan bahasa formal sederhana dan konsisten.
- Jika suatu variabel tidak dapat dilihat karena gambar pendukung tidak tersedia, gunakan label "TIDAK_TERIDENTIFIKASI".
- Jangan menggunakan bullet point, numbering, markdown, maupun simbol tambahan pada bagian reasoning.
- Output wajib mengikuti format output pada system prompt.
""".strip()

In [59]:
def build_user_content(record: dict, schema_mode: str, target: str = "sft") -> List[dict]:
    if target not in {"openrouter", "sft"}:
        raise ValueError("target harus 'openrouter' atau 'sft'.")

    house_type = (record.get("house_type") or "").lower().strip()
    ext_img, int_img, single_img = select_images(record)
    content: List[dict] = []

    def add_image(img_obj: dict, caption: str) -> None:
        img_path = image_path_from_metadata(img_obj["image_path"])

        # gambar dulu
        if target == "openrouter":
            content.append({
                "type": "image_url",
                "image_url": {"url": encode_image_to_data_url(img_path)}
            })
        else:
            content.append({
                "type": "image",
                "image": str(img_path)
            })

        # caption singkat setelah gambar, supaya tetap jelas konteksnya
        content.append({
            "type": "text",
            "text": caption
        })

    if house_type == "multi":
        if ext_img:
            add_image(ext_img, "Foto tampak luar.")
        if int_img:
            add_image(int_img, "Foto tampak dalam.")
    else:
        if single_img:
            vt = (single_img.get("view_type") or "exterior").lower().strip()
            caption = "Foto tampak luar." if vt == "exterior" else "Foto tampak dalam."
            add_image(single_img, caption)

    # prompt utama diletakkan paling akhir
    content.append({
        "type": "text",
        "text": build_user_prompt_text(record, schema_mode, target=target)
    })

    return content

In [60]:
def build_messages_for_api(record: dict, schema_mode: str) -> List[dict]:
    house_type = (record.get("house_type") or "").lower().strip()
    return [
        {"role": "system", "content": get_system_prompt("openrouter", schema_mode, house_type)},
        {"role": "user", "content": build_user_content(record, schema_mode, target="openrouter")},
    ]

In [61]:
def clean_assistant_response(text: Optional[str]) -> tuple[Optional[str], str]:
    """
    Validate and clean assistant response.
    Returns: (cleaned_text, reason) where reason explains any rejection
    """
    if not text:
        return None, "empty_response"

    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()

    lowered = cleaned.lower()
    if any(keyword in lowered for keyword in REFUSAL_KEYWORDS):
        return None, "refusal_keywords_detected"
    
    if len(cleaned) < 25:
        return None, f"too_short ({len(cleaned)} chars, min 25)"

    required_markers = ["Atap", "Dinding", "Lantai", "Reasoning", "Kesimpulan"]
    missing = [m for m in required_markers if m not in cleaned]
    if missing:
        return None, f"missing_markers: {missing}"

    return cleaned, "ok"

In [62]:
def get_system_prompt(target: str, schema_mode: str, house_type: str) -> str:
    house_type = (house_type or "").lower().strip()
    schema_mode = schema_mode.lower().strip()
    target = target.lower().strip()

    if target not in {"openrouter", "sft"}:
        raise ValueError("target harus 'openrouter' atau 'sft'.")

    if target == "openrouter":
        if schema_mode == "cnn_vlm" and house_type == "multi":
            return SYSTEM_PROMPT_CNNVLM_MULTI
        if schema_mode == "cnn_vlm" and house_type == "single":
            return SYSTEM_PROMPT_CNNVLM_SINGLE
        if schema_mode == "vlm_only" and house_type == "multi":
            return SYSTEM_PROMPT_VLMONLY_MULTI
        if schema_mode == "vlm_only" and house_type == "single":
            return SYSTEM_PROMPT_VLMONLY_SINGLE
        raise ValueError(f"Schema atau house_type tidak dikenal: {schema_mode} / {house_type}")

    if schema_mode == "cnn_vlm":
        return SYSTEM_PROMPT_SFT_CNN_VLM
    if schema_mode == "vlm_only":
        return SYSTEM_PROMPT_SFT_VLM_ONLY
    raise ValueError(f"DATASET_SCHEMA tidak dikenal: {schema_mode}")

In [63]:
def build_sft_sample(record: dict, schema_mode: str, assistant_text: Optional[str] = None) -> dict:
    house_type = (record.get("house_type") or "").lower().strip()
    messages = [
        {"role": "system", "content": get_system_prompt("sft", schema_mode, house_type)},
        {"role": "user", "content": build_user_content(record, schema_mode, target="sft")},
    ]
    if assistant_text is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": assistant_text}]})

    return {
        "id": {
            "house_id": record.get("house_id"),
            "source_group_id": record.get("source_group_id"),
            "house_type": record.get("house_type"),
            "split": record.get("split"),
        },
        "messages": messages,
    }

### OpenRouter Call

In [64]:
session = requests.Session()

In [65]:
def call_openrouter(record: dict, schema_mode: str) -> Optional[dict]:
    payload = {
        "model": MODEL_NAME,
        "messages": build_messages_for_api(record, schema_mode),
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,

        "reasoning": {
            "effort": "minimal"
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=TIMEOUT,
            )

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code}: {response.text[:1000]}"
                time.sleep(RETRY_SLEEP_SEC * attempt)
                continue

            data = response.json()

            try:
                choice = data.get("choices", [{}])[0]
                message = choice.get("message", {})

                print({
                    "house_id": record.get("house_id"),
                    "finish_reason": choice.get("finish_reason"),
                    "native_finish_reason": choice.get("native_finish_reason"),
                    "has_content": message.get("content") is not None,
                })

            except Exception:
                pass

            # return FULL response JSON
            return data

        except Exception as e:
            last_error = str(e)
            time.sleep(RETRY_SLEEP_SEC * attempt)

    print(
        f"[ERROR] OpenRouter gagal untuk "
        f"house_id={record.get('house_id')} | {last_error}"
    )

    return None

### Generation Pipeline

In [66]:
def _generate_one_record(record: dict, schema_mode: str) -> dict:
    if not is_valid_record_images(record):
        return {
            'status': 'skip',
            'reason': 'invalid_images',
            'record': record,
            'assistant_text': None,
        }

    data = call_openrouter(record, schema_mode)

    # gagal request/API
    if data is None:
        return {
            'status': 'fail',
            'reason': 'openrouter_failed',
            'record': record,
            'assistant_text': None,
        }

    try:
        choice = data.get("choices", [{}])[0]
        message = choice.get("message", {})

        raw_output = message.get("content")

        usage = data.get("usage", {})

        reasoning_tokens = (
            usage.get("completion_tokens_details", {})
                 .get("reasoning_tokens")
        )

    except Exception as e:
        return {
            'status': 'fail',
            'reason': f'extract_error: {str(e)}',
            'record': record,
            'assistant_text': None,
        }

    cleaned, validation_reason = clean_assistant_response(raw_output)

    # validation/parser fail
    if cleaned is None:
        return {
            'status': 'fail',
            'reason': validation_reason,
            'record': record,
            'assistant_text': raw_output,

            # metadata
            'finish_reason': choice.get("finish_reason"),
            'native_finish_reason': choice.get("native_finish_reason"),

            # usage
            'usage': usage,
            'reasoning_tokens': reasoning_tokens,

            # model info
            'model': data.get("model"),
            'provider': data.get("provider"),

            'reasoning': message.get("reasoning"),
        }

    # SUCCESS
    return {
        'status': 'ok',
        'reason': None,
        'record': record,
        'assistant_text': cleaned,

        # metadata
        'finish_reason': choice.get("finish_reason"),
        'native_finish_reason': choice.get("native_finish_reason"),
        
        # usage
        'usage': usage,
        'reasoning_tokens': reasoning_tokens,

        # model info
        'model': data.get("model"),
        'provider': data.get("provider"),

        'reasoning': message.get("reasoning"),
    }

In [67]:
def read_existing_ids(output_path: Path) -> set:
    done_ids = set()
    if not output_path.exists():
        return done_ids

    with open(output_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                meta = obj.get('id', {})
                key = (
                    meta.get('house_id'),
                    meta.get('source_group_id'),
                    meta.get('house_type'),
                    meta.get('split'),
                )
                done_ids.add(key)
            except Exception:
                continue
    return done_ids

In [68]:
def append_jsonl(path: Path, samples: List[dict]) -> None:
    if not samples:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        for sample in samples:
            f.write(json.dumps(sample, ensure_ascii=False) + '\n')

In [69]:
def generate_split(
    records: List[dict],
    split_name: str,
    output_path: Path,
    schema_mode: str,
    cache: Dict[str, dict],
    max_workers: int = MAX_WORKERS,
    use_openrouter: bool = True,
) -> List[dict]:
    done_ids = read_existing_ids(output_path)
    pending: List[dict] = []
    final_samples: List[dict] = []

    for record in records:
        key = (
            record.get('house_id'),
            record.get('source_group_id'),
            record.get('house_type'),
            record.get('split'),
        )
        if key in done_ids:
            continue

        if not is_valid_record_images(record):
            continue

        cache_key = sample_key(record)
        if use_openrouter and cache_key in cache:
            cached = cache[cache_key]
            if cached.get('status') == 'ok' and cached.get('assistant_text'):
                final_samples.append(build_sft_sample(record, schema_mode, cached['assistant_text']))
                continue

        pending.append(record)

    if use_openrouter and pending:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(_generate_one_record, record, schema_mode): record
                for record in pending
            }

            for future in tqdm(as_completed(futures), total=len(futures), desc=f'Generating {split_name}'):
                result = future.result()
                record = result['record']
                cache_key = sample_key(record)

                cache[cache_key] = {

                     # original record
                    'record': {
                        'house_id': record.get('house_id'),
                        'source_group_id': record.get('source_group_id'),
                        'house_type': record.get('house_type'),
                        'split': record.get('split'),
                    },

                    'status': result['status'],
                    'reason': result['reason'],

                    # assistant output
                    'assistant_text': result['assistant_text'],

                    # model info
                    'model': result.get('model'),
                    'provider': result.get('provider'),

                    # response metadata
                    'finish_reason': result.get('finish_reason'),
                    'native_finish_reason': result.get('native_finish_reason'),

                    # token usage
                    'usage': result.get('usage'),
                    'reasoning_tokens': result.get('reasoning_tokens'),

                    # reasoning
                    'reasoning': result.get('reasoning'),
                }

                if result['status'] == 'ok':
                    final_samples.append(build_sft_sample(record, schema_mode, result['assistant_text']))

    elif not use_openrouter:
        for record in pending:
            final_samples.append(build_sft_sample(record, schema_mode, assistant_text=None))

    final_samples = sorted(
        final_samples,
        key=lambda x: (
            x['id'].get('split', ''),
            x['id'].get('house_type', ''),
            x['id'].get('house_id', ''),
            x['id'].get('source_group_id', ''),
        ),
    )

    append_jsonl(output_path, final_samples)
    return final_samples

## Run Generation

In [70]:
cache = load_cache(CACHE_PATH)
print("Cache loaded:", len(cache))

train_records = sorted(records_by_split["train"], key=lambda r: (r.get("house_type", ""), r.get("house_id", "")))
val_records = sorted(records_by_split["val"], key=lambda r: (r.get("house_type", ""), r.get("house_id", "")))
test_records = sorted(records_by_split["test"], key=lambda r: (r.get("house_type", ""), r.get("house_id", "")))

train_samples = generate_split(
    train_records,
    "train",
    TRAIN_JSONL,
    DATASET_SCHEMA,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

val_samples = generate_split(
    val_records,
    "val",
    VAL_JSONL,
    DATASET_SCHEMA,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

test_samples = generate_split(
    test_records,
    "test",
    TEST_JSONL,
    DATASET_SCHEMA,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=False,
)

save_cache(CACHE_PATH, cache)

all_samples = []
for path in [TRAIN_JSONL, VAL_JSONL, TEST_JSONL]:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    all_samples.append(json.loads(line))

with open(ALL_JSONL, "w", encoding="utf-8") as f:
    for sample in all_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("\nSelesai.")
print("Train samples:", len(train_samples))
print("Val samples  :", len(val_samples))
print("Test samples :", len(test_samples))
print("All samples  :", len(all_samples))
print("Cache size   :", len(cache))
print("Output folder:", OUTPUT_BASE_DIR)

Cache loaded: 0


Generating train:  20%|██        | 1/5 [00:03<00:12,  3.24s/it]

{'house_id': 'H00039', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}


Generating train:  60%|██████    | 3/5 [00:03<00:01,  1.11it/s]

{'house_id': 'H00988', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}
{'house_id': 'H00185', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}


...

{'house_id': 'H00483', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}
{'house_id': 'H00228', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}
{'house_id': 'H00271', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}


Generating val:  80%|████████  | 4/5 [00:04<00:00,  1.35it/s]

{'house_id': 'H00433', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}


Generating val: 100%|██████████| 5/5 [00:06<00:00,  1.38s/it]

{'house_id': 'H00983', 'finish_reason': 'stop', 'native_finish_reason': 'completed', 'has_content': True}

Selesai.
Train samples: 5
Val samples  : 5
Test samples : 5
All samples  : 15
Cache size   : 10
Output folder: ..\data\sft_dataset\vlm_only
